# **投资组合优化的替代风险度量**

在投资组合优化中，传统的风险度量（如方差/标准差）有一个局限性：
它将高于预期收益的上行波动和低于预期收益的下行波动同等对待，但投资者通常只关心下行风险（损失）。

因此，为了更准确地反映投资者对损失的厌恶，人们开发了许多替代性的风险度量。这些度量主要侧重于下行波动和极端损失：

**风险度量 (Risk Measure)**
- 下偏矩 (Lower Partial Moments)：LPM,收益低于某一目标阈值（如无风险利率或零）的程度。
- 半方差 (Semivariance)：SV,收益低于均值的负向离差平方的期望值。它是 LPM 的一个特例（目标收益为均值，α=2）。
- 风险价值 (Value-at-Risk)：VaR,在给定置信水平下，投资组合可能遭受的最大预期损失。
- 条件风险价值 (Conditional Value-at-Risk) 或 预期短缺 (Expected Shortfall)：CVaR / ES,在损失超过 VaR 的情况下，平均损失的期望值。这是一种一致性风险度量。
- 最大回撤 (Maximum Drawdown)：MDD,从历史最高点到谷底的最大跌幅。衡量的是投资者在一定时期内可能经历的“痛苦”。

#### **下偏矩 (Lower Partial Moments, LPM) 的含义**
- 下偏矩是一种更关注下行风险的度量方法。
定义： 它衡量的是投资组合收益率 $R$ 低于某一目标收益率 $\tau$ 的负向偏离的期望值，并对这些偏离的 $\alpha$ 次方进行加权。

数学表达式：$$LPM_{\tau, \alpha} = E \left[ \max(0, \tau - R)^\alpha \right]$$其中：$E[\cdot]$ 是期望值。

- $\tau$ 是目标收益率（Target Return），例如无风险利率、零收益或投资组合的历史平均收益。
- $R$ 是投资组合的实际收益率。
- $\alpha$ 是投资者对下行风险的厌恶程度：当 $\alpha = 1$ 时，$LPM$ 衡量的是低于目标收益率的预期损失（也称为下行偏差/预期短缺）。
- 当 $\alpha = 2$ 时，$LPM$ 衡量的是下半方差（Semi-Variance），这相当于对低于目标的偏离进行平方加权。
  
**核心思想： 只有低于目标收益率 $\tau$ 的负向波动才被视为风险。这解决了传统方差将上行波动也视为风险的问题**。

#### **尾部风险 (Tail Risk) 和下行风险 (Downside Risk) 的定义**

**1. 下行风险 (Downside Risk)**
- 定义： 下行风险是指投资组合的收益率**低于某个预设目标或阈值（如零或无风险利率）**的可能性和程度。
- 特征： 它是对损失的衡量，只关注收益分布的左侧。下偏矩 (LPM) 就是一种广义的下行风险度量。

**2. 尾部风险 (Tail Risk)**
- 定义： 尾部风险是指收益分布中极端负向事件（即“肥尾”部分）发生的可能性及其影响程度。
- 特征：
    - 它关注的是分布远端的极端损失，这些损失发生的概率很低，但一旦发生，影响会非常巨大（例如金融危机中的大幅亏损）。
    - 与标准差关注的分布中部波动不同，尾部风险专注于分布左侧的极值。
    - 风险价值 (VaR) 和 条件风险价值 (CVaR/ES) 是衡量尾部风险最常用的工具。CVaR 尤其重要，因为它衡量的是“最坏情况”下的平均损失。

**简而言之，下行风险是一个更宽泛的概念，指所有低于目标收益的风险；而尾部风险特指那些极度罕见且损失巨大的下行风险**。

# **后悔调整协方差矩阵作为新的另类风险**

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from scipy.optimize import minimize
import math
from scipy.stats import skew
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文黑体
plt.rcParams['axes.unicode_minus'] = False    # 正常显示负号

In [3]:

def MVR_ret(mu, sigma_vol, corr_matrix, gamma=3, alpha=0.5, n_simulations=100000, seed=42):
    # 构建原始协方差矩阵 Cov(i, j) = rho * sigma_i * sigma_j
    N = len(mu)
    sigma_matrix = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            sigma_matrix[i, j] = corr_matrix[i, j] * sigma_vol[i] * sigma_vol[j]
            
    # 生成模拟收益数据，捕捉 R_max 的随机性，计算 Property 1 所需的新协方差矩阵
    rng = np.random.default_rng(seed)
    R = rng.multivariate_normal(mu, sigma_matrix, n_simulations)
    # 形状为 (n_simulations, 1)
    R_max = np.max(R, axis=1, keepdims=True)
    # 计算后悔调整后的收益序列 Z (Regret-adjusted Returns)
    Z = R - alpha * R_max
    # 计算后悔调整后的协方差矩阵，用 Z 的协方差矩阵替换原始矩阵
    Sigma_regret = np.cov(Z, rowvar=False) # rowvar=False 表示每一列是一个变量(资产)
    
    # 标准均值-方差优化求解 Max (mu.T * w - gamma * w.T * Sigma_new * w)
    # 等价于 Min (gamma * w.T * Sigma_new * w - mu.T * w)
    def objective(weights):
        Var = weights.T @ Sigma_regret @ weights
        Ret = weights.T @ mu
        return gamma * Var - Ret

    # 约束条件: 权重之和为 1
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(N))
    
    # 初始猜测
    init_guess = np.ones(N) / N
    result = minimize(objective, init_guess, method='SLSQP', bounds=bounds, constraints=constraints)
    
    return result.x, Sigma_regret

In [4]:
mu_vec = np.array([0.05, 0.15])      # 资产1: 5%, 资产2: 15%
sigma_vec = np.array([0.10, 0.40])   # 资产1: 10%, 资产2: 40%
corr_mat = np.array([[1.0, 0.5], 
                     [0.5, 1.0]])    # 相关系数 0.5
gamma_val = 3                        # 风险厌恶 3


# Markowitz (alpha = 0)
w_markowitz, cov_markowitz = MVR_ret(mu_vec, sigma_vec, corr_mat, gamma=gamma_val, alpha=0.0)
print(f"【Markowitz Case (alpha=0)】")
print(f"协方差矩阵: {cov_markowitz[0,0]:.6f}")
print(f"最优权重: Asset 1 = {w_markowitz[0]:.2%}, Asset 2 = {w_markowitz[1]:.2%}")

# Pure Regret - Return View (alpha = 1)
w_regret, cov_regret = MVR_ret(mu_vec, sigma_vec, corr_mat, gamma=gamma_val, alpha=1.0)
print(f"【Pure Regret Case (alpha=1)】")
print(f"协方差矩阵: {cov_regret[0,0]:.6f}")
print(f"最优权重: Asset 1 = {w_regret[0]:.2%}, Asset 2 = {w_regret[1]:.2%}")

【Markowitz Case (alpha=0)】
协方差矩阵: 0.010074
最优权重: Asset 1 = 94.71%, Asset 2 = 5.29%
【Pure Regret Case (alpha=1)】
协方差矩阵: 0.059660
最优权重: Asset 1 = 26.08%, Asset 2 = 73.92%


In [ ]:
def calculate_pi_max_2_assets_vectorized(R, sigma_matrix, gamma):
    r1 = R[:, 0]
    r2 = R[:, 1]
    s1_sq = sigma_matrix[0, 0]
    s2_sq = sigma_matrix[1, 1]
    s12 = sigma_matrix[0, 1]
    
    # 效用函数 U(w) = w(r1-r2) + r2 - gamma*[w^2*s1^2 + (1-w)^2*s2^2 + 2w(1-w)s12]
    # 对 w 求导令为 0:
    # dU/dw = (r1-r2) - gamma * [2w*s1^2 - 2(1-w)*s2^2 + 2(1-2w)*s12] = 0
    # (r1-r2) - 2*gamma * [ w(s1^2 + s2^2 - 2s12) + (s12 - s2^2) ] = 0
    
    A = 2 * gamma * (s1_sq + s2_sq - 2 * s12)
    B = 2 * gamma * (s12 - s2_sq)
    C = r1 - r2
    
    # w_star * A + B = C  =>  w_star = (C - B) / A
    # 注意 A 通常 > 0 (只要不是完全相关且波动率相等)
    w_star = (C - B) / A
    
    # 考虑不允许卖空限制 w 属于 [0, 1]
    w_star = np.clip(w_star, 0.0, 1.0)
    
    # 计算对应的最优效用值 Pi_max
    w_star = w_star.reshape(-1, 1)
    
    # Portfolio Return: w*r1 + (1-w)*r2
    port_ret = w_star * r1.reshape(-1, 1) + (1 - w_star) * r2.reshape(-1, 1)
    
    # Portfolio Variance: w^2 s1^2 + ... (Ex-ante variance, constant for specific w)
    # 注意：这里每一行 w 不同，所以方差也是动态计算的
    port_var = (w_star**2 * s1_sq + (1 - w_star)**2 * s2_sq + 2 * w_star * (1 - w_star) * s12)
    
    pi_max = port_ret - gamma * port_var
    return pi_max

In [5]:
def get_preference_regret_weights(mu, sigma_vol, corr_matrix, gamma=3, alpha=0.5, n_simulations=100000, seed=42):
    N = len(mu)
    # 构建原始协方差矩阵
    sigma_matrix = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            sigma_matrix[i, j] = corr_matrix[i, j] * sigma_vol[i] * sigma_vol[j]      
    # 模拟数据
    rng = np.random.default_rng(seed)
    R = rng.multivariate_normal(mu, sigma_matrix, n_simulations)
    
    # 计算 Pi_max (事后最优偏好值)
    # 针对 2 资产情形，使用向量化解析解加速。
    if N == 2:
        pi_max = calculate_pi_max_2_assets_vectorized(R, sigma_matrix, gamma)
    else:
        # 如果资产大于2，通过简化假设或循环求解（较慢）
        # 这里为了复现演示，仅支持2资产快速计算
        pi_max = np.zeros((n_simulations, 1))
        # (此处省略多资产的复杂循环实现)

    # 计算变量 Z (Regret-adjusted variable)
    # Z = R - alpha * pi_max
    # 注意 pi_max 是一个标量序列，广播到每个资产上
    Z = R - alpha * pi_max
    
    # 计算 Sigma_ra (Z 的协方差)
    Sigma_ra_pref = np.cov(Z, rowvar=False)
    
    # 构建最终的有效协方差矩阵
    Sigma_total = alpha * sigma_matrix + Sigma_ra_pref
    
    # 7. 优化求解
    # 使用原始 gamma 和 新矩阵 Sigma_total
    def objective(weights):
        Var = weights.T @ Sigma_total @ weights
        Ret = weights.T @ mu
        return gamma * Var - Ret

    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(N))
    init_guess = np.ones(N) / N
    
    result = minimize(objective, init_guess, method='SLSQP', bounds=bounds, constraints=constraints)
    
    return result.x, Sigma_total

In [7]:
# 使用同样的参数
mu_vec = np.array([0.05, 0.15])
sigma_vec = np.array([0.10, 0.40])
corr_mat = np.array([[1.0, 0.5], [0.5, 1.0]])
gamma_val = 3

w_pref, cov_pref = get_preference_regret_weights(mu_vec, sigma_vec, corr_mat, gamma=gamma_val, alpha=1.0)

print(f"【Preference Regret Case (alpha=1)】")
print(f"协方差矩阵 (左上角): {cov_pref[0,0]:.6f}")
print(f"最优权重: Asset 1 = {w_pref[0]:.2%}, Asset 2 = {w_pref[1]:.2%}")

【Preference Regret Case (alpha=1)】
协方差矩阵 (左上角): 0.020245
最优权重: Asset 1 = 86.96%, Asset 2 = 13.04%


In [ ]:
np.random.seed(42)
# ---- 参数（与论文 base scenario 对齐） ----
mu = np.array([0.10, 0.10])        # 期望（年化或单位一致）
sigma = np.array([0.25, 0.25])     # 标准差
rho = 0.5
gamma = 3.0
alpha = 1.0                        # 纯 regret case (alpha=1)；可改为 0.5, 0
n_sims = 200_000                   # 试验用 200k；论文用了 2,000,000

# 构造协方差矩阵
cov = np.array([[sigma[0]**2, rho*sigma[0]*sigma[1]],[rho*sigma[0]*sigma[1], sigma[1]**2]])

# 生成多元正态模拟（每次模拟是两资产的 realized returns）
R = np.random.multivariate_normal(mean=mu, cov=cov, size=n_sims)  # shape (n_sims, 2)

# ---- return-regret: Z_ret_{j,t} = R_{j,t} - alpha * R_max_t ----
R_max = R.max(axis=1)                         # 每次模拟的 ex-post 最佳资产收益
Z_ret = R - alpha * R_max[:, None]            # 广播，shape (n_sims,2)

# 估计 regret-adjusted 均值与协方差
mu_ra_ret = Z_ret.mean(axis=0)
cov_ra_ret = np.cov(Z_ret, rowvar=False, bias=False)

# ---- preference-regret:
# 论文中 π(R_j, sigma_j^2) ~ R_j - gamma * sigma_j^2  （使用已知 sigma_j^2）
# π_max_t = max_j [ R_{j,t} - gamma * sigma_j^2 ]
pi_vals = R - gamma * (sigma**2)[None, :]     # shape (n_sims,2)
pi_max = pi_vals.max(axis=1)
# Z_pref_{j,t} = R_{j,t} - alpha * pi_max_t - alpha * gamma * sigma_j^2
Z_pref = R - alpha * pi_max[:, None] - alpha * gamma * (sigma**2)[None, :]

mu_ra_pref = Z_pref.mean(axis=0)
cov_ra_pref = np.cov(Z_pref, rowvar=False, bias=False)

# ---- 求解带预算约束的 μ-σ 最优权重（闭式，无短售约束）
# 论文给的通用 closed-form 推导。我们用直接线性代数实现：
def mean_variance_weights(mu_vec, cov_mat, gamma):
    # 目标： max_w w^T mu - gamma * w^T cov w ; subject to 1^T w = 1
    # 解法（见论文/附录）： w = (1/(2γ)) Σ^{-1} μ  - (λ/(2γ)) Σ^{-1} 1
    # λ 通过预算约束求解： λ = (1^T Σ^{-1} μ - 2γ) / (1^T Σ^{-1} 1)
    invS = np.linalg.inv(cov_mat)
    A = invS.dot(mu_vec)
    B = invS.dot(np.ones_like(mu_vec))
    denom = np.ones_like(mu_vec).dot(B)
    lam = (np.ones_like(mu_vec).dot(A) - 2*gamma) / denom
    w = (A - lam * B) / (2*gamma)
    return w

w_markowitz = mean_variance_weights(mu, cov, gamma)
w_ret = mean_variance_weights(mu_ra_ret, cov_ra_ret, gamma)
w_pref = mean_variance_weights(mu_ra_pref, cov_ra_pref, gamma)

# 输出对照
print("Markowitz weights:", w_markowitz)
print("Pure return-regret weights:", w_ret)
print("Pure preference-regret weights:", w_pref)

# 还可以计算组合的事前分布（均值、std、skew、ES等）用于表格对比（见论文 Table）


### **Property 1 的核心结论**

Property 1 指出：最优投资组合的权重可以通过标准的 $\mu-\sigma$ 优化程序获得，
- **其中：期望收益向量：保持不变，使用原始的 $\mu$。协方差矩阵：使用调整后的新矩阵**。
- Return-Regret (收益-后悔) 视角：$$\Sigma^{new} = \Sigma_{ra}^{ret}$$
- Preference-Regret (偏好-后悔) 视角：$$\Sigma^{new} = \alpha \Sigma + \Sigma_{ra}^{pref}$$

#### **Return-Regret (收益-后悔) 视角的推导**

- 后悔调整后的收益 $Z_P^{ret}$ 定义为 2：$$Z_P^{ret} = R_P - \alpha R_{max}$$
    - 其中 $R_P = \omega^{tr}R$ 是投资组合收益，$R_{max}$ 是事后表现最好的单一资产收益。

- 目标函数：投资者最大化后悔调整后收益的均值-方差效用：$$\Pi(Z_P^{ret}) = E[Z_P^{ret}] - \gamma Var(Z_P^{ret})$$

**1. 步骤 A：展开期望项 $E[\cdot]$**：
$$E[Z_P^{ret}] = E[\omega^{tr}R - \alpha R_{max}] = \omega^{tr}\mu - \alpha E[R_{max}]$$
- 关键点：$\alpha E[R_{max}]$ 是一个与权重 $\omega$ 无关的常数（一旦资产池确定，$R_{max}$ 的期望就是固定的）。
- 因此，该项为 0。优化 $\omega^{tr}\mu - C$ 等价于优化 $\omega^{tr}\mu$。

**2. 步骤 B：展开方差项 $Var(\cdot)$**：
$$Var(Z_P^{ret}) = Var(\omega^{tr}R - \alpha R_{max})$$
- 由于 $R_{max}$ 是所有资产收益的函数，它与 $R$ 存在相关性。我
- 们将 $Z^{ret}$ 的协方差矩阵定义为 $\Sigma_{ra}^{ret}$。
- 那么组合的方差可以写成二次型：$$Var(Z_P^{ret}) = \omega^{tr} \Sigma_{ra}^{ret} \omega$$
    - 这里，$\Sigma_{ra}^{ret}$ 的第 $(i,j)$ 个元素为 ：$$(\Sigma_{ra}^{ret})_{i,j} = Cov(Z_i, Z_j) = Cov(R_i - \alpha R_{max}, R_j - \alpha R_{max})$$

**结论：将上述两步代入目标函数**：
$$\max_{\omega} \left( \underbrace{\omega^{tr}\mu}_{\text{原始均值部分}} - \underbrace{\alpha E[R_{max}]}_{\text{常数，丢弃}} - \gamma \underbrace{(\omega^{tr} \Sigma_{ra}^{ret} \omega)}_{\text{新方差部分}} \right)$$
这等价于标准的 Markowitz 问题：$\max (\omega^{tr}\mu - \gamma \omega^{tr} \tilde \Sigma \omega)$，其中 $\tilde \Sigma = \Sigma_{ra}^{ret}$。

In [40]:
T = 100
N = 5
R = np.random.randn(T, N)

def MVR_cov_ret(R, alpha):
    # 每个时期的事后最佳资产收益率
    R_max = R.max(axis=1)            # 形状 (T,)

    # 后悔调整后的收益
    Z = R - alpha * R_max[:, None]   # 形状 (T, N)

    # 后悔调整后收益的协方差
    Sigma_ra = np.cov(Z, rowvar=False, ddof=1)

    return Sigma_ra

In [41]:
def MVR_cov_ret_expanded(R, alpha):
    T, N = R.shape
    ones = np.ones((N, 1))

    Sigma = np.cov(R, rowvar=False, ddof=1)
    R_max = R.max(axis=1)

    # Cov(R, R_max)
    cov_full = np.cov(np.column_stack([R, R_max]), rowvar=False, ddof=1)
    cov_R_Rmax = cov_full[:N, -1]    # shape (N,)

    # Var(R_max)
    var_Rmax = np.var(R_max, ddof=1)

    # Expanded regret covariance
    Sigma_ra = (Sigma - alpha * (cov_R_Rmax[:, None] @ ones.T + ones @ cov_R_Rmax[None, :]) + alpha**2 * var_Rmax * (ones @ ones.T))

    return Sigma_ra

In [42]:
Sigma_A = MVR_cov_ret(R, alpha=1)
Sigma_B = MVR_cov_ret_expanded(R, alpha=1)

In [43]:
# 检验 1：两种实现完全一致
np.allclose(Sigma_A, Sigma_B, atol=1e-10)

True

In [44]:
# 检验 2：α = 0 退化为 Markowitz
Sigma_0 = MVR_cov_ret(R, alpha=0.0)
np.allclose(Sigma_0, np.cov(R, rowvar=False, ddof=1))

True

In [49]:
# 检验 3：对称 & 正定
np.allclose(Sigma_A, Sigma_A.T)
np.linalg.eigvalsh(Sigma_A).min() > 0

np.True_

#### **Preference-Regret (偏好-后悔) 视角的推导**

- 后悔定义为“事后最优偏好值”与“实际偏好值”之差。
- 目标函数：原始目标是最大化 $Z_P^{pref}$ 的效用。
- 将其展开为：
- $$\Pi(Z_P^{pref}) = \mu_P - \gamma v - \alpha E[\pi_{max}]$$
    - 这里 $\mu_P = \omega^{tr}\mu$。
    - 常数项：$\alpha E[\pi_{max}]$ 是事后最优偏好值的期望，与当前选择的 $\omega$ 无关，丢弃。
    - 风险项 $v$：这是推导的难点。定义了一个“调整后的总风险” $v$：
$$v = Var(R_P - \alpha \pi_{max}) + \alpha Var(R_P)$$
    - (注：这里 $R_P$ 是组合收益，$\pi_{max}$ 是事后最优组合产生的偏好值，它是一个随机变量)

步骤：矩阵化表达 $v$我们需要把 $v$ 写成 $\omega^{tr} (\dots) \omega$ 的形式。
1. 第一部分：$Var(R_P - \alpha \pi_{max})$令 $\Sigma_{ra}^{pref}$ 为随机变量向量 $(R - \alpha \pi_{max})$ 的协方差矩阵。则 $Var(R_P - \alpha \pi_{max}) = Var(\omega^{tr}R - \alpha \pi_{max}) = Var(\omega^{tr}(R - \alpha \pi_{max} \cdot \mathbf{1}))$。
- 这对应了矩阵二次型：$$\omega^{tr} \Sigma_{ra}^{pref} \omega$$
2. 第二部分：$\alpha Var(R_P)$这直接对应原始协方差矩阵：$$\alpha (\omega^{tr} \Sigma \omega) = \omega^{tr} (\alpha \Sigma) \omega$$

**结论：将两部分合并**：$$v = \omega^{tr} \Sigma_{ra}^{pref} \omega + \omega^{tr} (\alpha \Sigma) \omega = \omega^{tr} (\Sigma_{ra}^{pref} + \alpha \Sigma) \omega$$
**因此，新的有效协方差矩阵为**：$$\tilde{\Sigma}^{pref} = \alpha \Sigma + \Sigma_{ra}^{pref}$$

In [45]:
def MVR_cov_pref(R, alpha, gamma):
    T, N = R.shape

    # 初始协方差矩阵
    Sigma = np.cov(R, rowvar=False, ddof=1)

    # 事前方差 
    var_assets = np.var(R, axis=0, ddof=1)   # 形状 (N,)

    # 事后偏好值
    # pi_{i,t} = R_{i,t} - gamma * sigma_i^2
    pi = R - gamma * var_assets[None, :]     # 形状 (T, N)

    # 事后最优偏好
    pi_max = pi.max(axis=1)                  # 形状 (T,)

    # 后悔调整随机分量
    Z_pref = R - alpha * pi_max[:, None]

    # 后悔调整后收益的协方差
    Sigma_ra_pref = np.cov(Z_pref, rowvar=False, ddof=1)

    # Property 1
    Sigma_pref = alpha * Sigma + Sigma_ra_pref

    return Sigma_pref

| 论文对象                                      | 代码                              |
| ----------------------------------------- | ------------------------------- |
| $\Sigma$                                  | `Sigma`                         |
| $\pi_{i,t} = R_{i,t} - \gamma \sigma_i^2$ | `pi = R - gamma * var_assets`   |
| $\pi_{\max,t}$                            | `pi.max(axis=1)`                |
| $\Sigma^{pref}_{ra}$                      | `np.cov(Z_pref)`                |
| $\tilde{\Sigma}^{pref}$                   | `alpha * Sigma + Sigma_ra_pref` |


In [46]:
# 检验 1: α = 0 退化为 Markowitz是否为 True
Sigma_pref_0 = MVR_cov_pref(R, alpha=0.0, gamma=3)
np.allclose(Sigma_pref_0, np.cov(R, rowvar=False, ddof=1))

True

In [47]:
# 检验 2: 对称性 & 正定性
Sigma_pref = MVR_cov_pref(R, alpha=1, gamma=3)

np.allclose(Sigma_pref, Sigma_pref.T)
np.linalg.eigvalsh(Sigma_pref).min() > 0

np.True_

In [48]:
# 检验 3: gamma → 0 时，检查preference-regret → return-regret
Sigma_pref_small_gamma = MVR_cov_pref(R, alpha=1, gamma=1e-8)

Sigma_ret = MVR_cov_ret(R, alpha=1)
alpha = 1
np.allclose(Sigma_pref_small_gamma, alpha * np.cov(R, rowvar=False) + Sigma_ret,atol=1e-6)

True

In [ ]:
# 计算每期的R_max（事后最佳资产收益）
df_assets['R_max'] = df_assets[['Mkt-RF', 'SMB', 'HML', 'MOM']].max(axis=1)
cov_matrix_daily = df_assets[['Mkt-RF', 'SMB', 'HML', 'MOM', 'R_max']].cov()
cov_with_Rmax_daily = cov_matrix_daily.loc[['Mkt-RF', 'SMB', 'HML', 'MOM'], 'R_max']
cov_with_Rmax_annual = cov_with_Rmax_daily * annual_factor  # 252
# 计算后悔调整后收益率 Z_i^{ret} = R_i - R_max (α=1)
Z_ret = df_assets[['Mkt-RF', 'SMB', 'HML', 'MOM']].subtract(df_assets['R_max'], axis=0)
Z_ret_std_annual = Z_ret.std() * np.sqrt(annual_factor)

# 创建收益视角后悔部分的数据框
regret_ret_data = []
assets = ['Mkt-RF', 'SMB', 'HML', 'MOM']

# Regret adjustment μ 行
regret_adj_values = cov_with_Rmax_annual[assets].round(4)
regret_ret_data.append(regret_adj_values)
# 偏差行
deviations = regret_adj_values - regret_adj_values.mean()
regret_ret_data.append(pd.Series([f"({dev:.4f})" for dev in deviations], index=assets))

# Regret-adj. std. dev. 行
regret_std_values = Z_ret_std_annual[assets].round(4)
regret_ret_data.append(regret_std_values)
# 偏差行
deviations_std = regret_std_values - regret_std_values.mean()
regret_ret_data.append(pd.Series([f"({dev:.4f})" for dev in deviations_std], index=assets))

regret_ret_df = pd.DataFrame(regret_ret_data, index=['Regret adjustment μ', '', 'Regret-adj. std. dev.', ''], columns=assets)
regret_ret_df